# 04 — Consistência das atividades: o mesmo tipo em dois idiomas

**Dimensão:** consistência (ver `AGENTS.md`, seção "Qualidade de dados").

**Objetivo:** ver se a mesma categoria aparece escrita de mais de um jeito na coluna `tipo`.

**Pedido:**
1. listar os valores de `tipo`, com a contagem de cada um;
2. cruzar `tipo` com `origem`;
3. apontar os valores que parecem a mesma categoria e propor um mapa de padronização, **sem aplicar**;
4. mostrar quantas linhas cada troca afetaria.

**Referência (roteiro do professor):** 11 valores para 7 tipos (corrida e run, caminhada e walk, forca e strength, pedal e ride); os nomes em inglês (376.930 treinos, 30%) vêm todos da origem `app`. Armadilha: contar só `corrida` deixa de fora 162.108 corridas.

Nada em `dados/brutos/` é alterado.

In [ ]:
import pandas as pd
from pathlib import Path

BRUTOS = Path('../dados/brutos')
atividades = pd.read_parquet(BRUTOS / 'atividades.parquet', columns=['atividade_id', 'tipo', 'origem', 'inicio'])
total = len(atividades)

## 1. Valores de `tipo`

In [ ]:
contagem = atividades['tipo'].value_counts()
print(f'{contagem.size} valores distintos')
contagem

## 2. `tipo` × `origem`

In [ ]:
cruzamento = pd.crosstab(atividades['tipo'], atividades['origem'], margins=True, margins_name='total')
cruzamento

## 3 e 4. Mapa de padronização e linhas afetadas

Os quatro nomes em inglês têm um par em português com o mesmo significado. A proposta é padronizar em **português**, o idioma das outras origens e do dicionário. O mapa é só uma proposta: não é aplicado aqui, e quando for, será em `dados/processados/`.

In [ ]:
MAPA_TIPO = {'run': 'corrida', 'walk': 'caminhada', 'strength': 'forca', 'ride': 'pedal'}

trocas = pd.DataFrame({
    'de': list(MAPA_TIPO),
    'para': list(MAPA_TIPO.values()),
    'linhas_afetadas': [contagem[t] for t in MAPA_TIPO],
    'linhas_ja_no_destino': [contagem[t] for t in MAPA_TIPO.values()],
})
trocas['total_depois'] = trocas['linhas_afetadas'] + trocas['linhas_ja_no_destino']
trocas['% do tipo que estava em inglês'] = (trocas['linhas_afetadas'] / trocas['total_depois'] * 100).round(1)
trocas

In [ ]:
tipos_padronizados = atividades['tipo'].replace(MAPA_TIPO).nunique()
em_ingles = atividades['tipo'].isin(MAPA_TIPO.keys())
origens_ingles = atividades.loc[em_ingles, 'origem'].unique().tolist()

print(f'{contagem.size} valores → {tipos_padronizados} tipos')
print(f'Treinos com nome em inglês: {em_ingles.sum():,} ({em_ingles.mean():.0%}) | origens: {origens_ingles}')

assert contagem.size == 11 and tipos_padronizados == 7
assert em_ingles.sum() == 376_930 and round(em_ingles.mean() * 100) == 30
assert origens_ingles == ['app']

## A armadilha: contar só `corrida`

In [ ]:
so_corrida = contagem['corrida']
corridas_de_verdade = contagem['corrida'] + contagem['run']
print(f'Só corrida: {so_corrida:,} | corrida + run: {corridas_de_verdade:,} | ficam de fora: {contagem["run"]:,}')
assert contagem['run'] == 162_108

## Além do roteiro: quando o inglês começou

Se o inglês vem só do `app`, a pergunta seguinte é desde quando. A coluna `inicio` responde.

In [ ]:
PARES = list(MAPA_TIPO) + list(MAPA_TIPO.values())
app = atividades[(atividades['origem'] == 'app') & atividades['tipo'].isin(PARES)].copy()
app['dia'] = app['inicio'].str[:10]
app['mes'] = app['inicio'].str[:7]
app['em_ingles'] = app['tipo'].isin(MAPA_TIPO.keys())

primeiro_dia = app.loc[app['em_ingles'], 'dia'].min()
print(f'Primeiro treino com nome em inglês: {primeiro_dia}')
assert primeiro_dia == '2026-04-15'

por_mes = app.groupby('mes')['em_ingles'].mean().mul(100).round(1).rename('% em inglês (app, tipos pareados)')
por_mes

In [ ]:
por_dia = app.groupby('dia')['em_ingles'].mean().mul(100).round(1)
por_dia['2026-04-10':'2026-04-20']

In [ ]:
depois = app[app['dia'] >= primeiro_dia]
residuo_pt = (~depois['em_ingles']).sum()
print(f'Depois de {primeiro_dia}, no app: {depois["em_ingles"].mean():.1%} em inglês; {residuo_pt:,} ainda em português')
depois.loc[~depois['em_ingles'], 'mes'].value_counts().sort_index()

A troca é abrupta: zero treinos em inglês até 14/04/2026 e cerca de 99% dos treinos pareados do app em inglês a partir de 15/04. Isso tem cara de **versão nova do app** que passou a gravar o tipo em inglês. O resíduo em português (~1%), estável mês a mês, **são cópias de treinos do relógio**: o app importa o treino do Garmin ou do Apple Health mantendo o nome original. Verificado no notebook 06 (unicidade): os 4.032 treinos são todos cópias.

## Conclusão

- **Roteiro conferido:** 11 valores para 7 tipos; os 376.930 treinos em inglês (30%) vêm todos da origem `app`; contar só `corrida` deixa de fora 162.108 corridas.
- **Mapa proposto (não aplicado):** `run` → `corrida`, `walk` → `caminhada`, `strength` → `forca`, `ride` → `pedal`. Afeta 162.108, 88.509, 85.548 e 40.765 linhas.
- **Causa provável:** a mudança começa em **15/04/2026**, de um dia para o outro, só no `app`. Nada antes disso.
- **Ligação com o case:** abril de 2026 foi o pior mês de churn (8,1%, enunciado). Uma versão nova do app na mesma quinzena vira hipótese para o objetivo 2. Cruzamento aponta suspeito, não prova causa; o próximo passo é cruzar com `versao_app` e com os chamados de suporte do período.

Decisões registradas em `decisoes.md`.